In [390]:
import os, hashlib, binascii

Sources for Code: https://drive.google.com/file/d/1UmBmKrm7ax-xWpIIlP-cmKEkDqA5SqnI/view?pli=1

In [391]:
def crand(seed):
    r=[]
    r.append(seed)
    for i in range(30):
        r.append((16807 * r[-1]) % 2147483647)
        if r[-1] < 0:
            r[-1] += 2147483647
    for i in range(31, 34):
        r.append(r[len(r)-31])
    for i in range(34, 344):
        r.append((r[len(r)-31] + r[len(r)-3]) % 2**32)
    while True:
        next = r[len(r)-31]+r[len(r)-3] % 2**32
        r.append(next)
        yield (next >> 1 if next < 2**32 else (next % 2**32) >> 1)

mygen = crand(2026)
firstfour = [next(mygen) for i in range(4)]
    
print(firstfour)

[1199659537, 1872465372, 1381520923, 36027969]


In [ ]:
mygen = crand(2026)
rands = [next(mygen) for i in range(6)]
plaintext = b"Hello Cryptography"
hexplain = binascii.hexlify(plaintext)
hexkey = "".join(map(lambda x: format(x, 'x')[-6:], rands)) 
cipher_as_int = int(hexplain, 16) ^ int(hexkey, 16)
cipher_as_hex = format(cipher_as_int, 'x')

print("Cipher as int:", cipher_as_int)
print("Cipher as hex:", cipher_as_hex)

Cipher as int: 17531174702000424339215974891877867004758938
Cipher as hex: c93f7df7e2fc1b246255ca2ebc3f1fb20b9a


In [394]:
cipher_hex = "c93f7df7e2fc1b246255ca2ebc3f1fb20b9a"

# Generate keystream using derived seed
num_rands = len(cipher_hex) // 6
mygen = crand(2026)
rands = [next(mygen) for _ in range(num_rands)]
hexkey = "".join(map(lambda x: format(x, 'x')[-6:], rands))

plain_as_int = int(cipher_hex, 16) ^ int(hexkey, 16) # XOR ciphertext with key to get plaintext

plain_as_hex = format(plain_as_int, 'x')
if len(plain_as_hex) % 2 != 0:
    plain_as_hex = '0' + plain_as_hex

# plaintext = binascii.unhexlify(plain_as_hex).decode('utf-8')
plaintext = binascii.unhexlify(plain_as_hex).decode('utf-8')
print("Plaintext:", plaintext)

Plaintext: Hello Cryptography


In [395]:
nonce = b"cc4304c09aee"
hexnonce = binascii.hexlify(nonce)
oursecret = 54321

concatenated_hex = hexnonce + format(oursecret, 'x').encode()
even_length = concatenated_hex.rjust(len(concatenated_hex) + len(concatenated_hex) % 2, b'0')

hexhash = hashlib.sha256(binascii.unhexlify(even_length)).hexdigest()
newseed = (int(hexhash, 16)) % 2**32
print(newseed)


3336748862


In [396]:
import binascii, hashlib


full_hex = "3e08816f1377f89f1c596fc197dd52946c92577bfd7c25c3"

# Split nonce (first 12 hex chars = 6 bytes) from ciphertext
hexnonce = full_hex[:12].encode()
cipher_hex = full_hex[12:]

# Derive new seed using same method
oursecret = 61983
concatenated_hex = hexnonce + format(oursecret, 'x').encode()
even_length = concatenated_hex.rjust(len(concatenated_hex) + len(concatenated_hex) % 2, b'0')
hexhash = hashlib.sha256(binascii.unhexlify(even_length)).hexdigest()
newseed = int(hexhash, 16) % 2**32
print("Derived seed:", newseed)

# Generate keystream using derived seed
num_rands = len(cipher_hex) // 6
mygen = crand(newseed)
rands = [next(mygen) for _ in range(num_rands)]
hexkey = "".join(map(lambda x: format(x, 'x')[-6:], rands))

# XOR to decrypt
plain_as_int = int(cipher_hex, 16) ^ int(hexkey, 16)
plain_as_hex = format(plain_as_int, 'x')
if len(plain_as_hex) % 2 != 0:
    plain_as_hex = '0' + plain_as_hex

plaintext = binascii.unhexlify(plain_as_hex).decode('utf-8')
print("Plaintext:", plaintext)

Derived seed: 42847799
Plaintext: this is a message.
